# 🐖 16 — Sensibilidad de calibración probabilística (CPU)

**Propósito:** resolver la segunda observación crítica del revisor. Reutiliza directamente las 37.936 detecciones etiquetadas en Notebook 14. Mide ECE con 5, 10, 15 y 20 bins, de igual anchura y de igual masa; todas las detecciones frente a las retenidas por `score_raw >= 0.25`. Importante: en la comparación antes/después el filtro se mantiene fijo sobre la confianza **sin calibrar**. No vuelve a ajustar Platt ni reentrena redes.

## 1. Entorno CPU

In [ ]:
%pip -q install "pandas>=2,<3" numpy "openpyxl>=3.1" matplotlib

## 2. Validar protocolos COCO y disponer de rutas

In [ ]:
from google.colab import drive,files
from pathlib import Path
from collections import Counter,defaultdict
from datetime import datetime,timezone
import json,zipfile,hashlib,math
import numpy as np,pandas as pd,matplotlib.pyplot as plt
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖')
assert ROOT.is_dir(), f'No encuentro {ROOT}; revisa la cuenta de Google'
REPORT=ROOT/'03_Resultados'/'05_Reportes'; METRIC=ROOT/'03_Resultados'/'03_Metricas'
FIG=ROOT/'03_Resultados'/'04_Figuras'; EXPORT=ROOT/'05_Exportaciones_ZIP'
for d in (REPORT,METRIC,FIG,EXPORT):d.mkdir(parents=True,exist_ok=True)
D09=REPORT/'CERDOS_09_COCO_Armonizado';D13=REPORT/'CERDOS_13_Calibracion_Validacion'
Y='YOLO11n-seg';M='Mask R-CNN'
def readj(p):
    assert p.is_file(),f'No encuentro: {p}'
    return json.loads(p.read_text(encoding='utf8'))
GT={'val':readj(D13/'CERDOS_13_GT_val_coco.json'),'test':readj(D09/'CERDOS_09_GT_test_coco.json')}
PREDS={
 ('val',Y):readj(D13/'CERDOS_13_Pred_YOLO_val_coco.json'),
 ('val',M):readj(D13/'CERDOS_13_Pred_MaskRCNN_val_coco.json'),
 ('test',Y):readj(D09/'CERDOS_09_Pred_YOLO_coco.json'),
 ('test',M):readj(D09/'CERDOS_09_Pred_MaskRCNN_coco.json')}
assert len(GT['val']['images'])==196 and len(GT['test']['images'])==373
assert len(GT['val']['annotations'])==1995 and len(GT['test']['annotations'])==4648
P09=readj(D09/'CERDOS_09_Protocolo.json');P13=readj(D13/'CERDOS_13_Protocolo.json')
for key in ('yaml_sha256','yolo_sha256','mask_sha256'):
    assert P09[key]==P13[key],f'Protocolos distintos: {key}'
print('✅ Predicciones previas y pesos verificados. No se necesita GPU ni reentrenar.')


## 3. Evaluación de sensibilidad sin contaminar el ajuste

In [ ]:
# Se parte de las etiquetas de detección y Platt registrados en el notebook 14.
SCOREFILE=METRIC/'CERDOS_14_Calibracion_Probabilistica.xlsx'
assert SCOREFILE.is_file(),'Ejecuta primero Notebook 14; no se modificarán sus resultados.'
D=pd.read_excel(SCOREFILE,sheet_name='Detecciones_Prob')
assert len(D)==37936 and set(D['modelo'])=={Y,M}
assert set(D['particion'])=={'validacion','prueba'}
assert D['correcta_IoU05'].isin([0,1]).all()
# Filtrar SIEMPRE por score original para comparar la misma población antes y después de Platt.
EVALS=[('Todas',.001),('Operativa_raw>=0.25',.25)]
BINS=(5,10,15,20)
def ece(y,p,k,method):
    p=np.clip(np.asarray(p,dtype=float),1e-6,1-1e-6);y=np.asarray(y,dtype=float)
    if method=='Igual_anchura':binid=np.minimum((p*k).astype(int),k-1)
    else:
        # Bines de tamaño igual (tamaño difiere a lo sumo en 1),
        # empates por orden estable: detalle en limitaciones.
        binid=np.empty(len(p),dtype=int)
        for j,ids in enumerate(np.array_split(np.argsort(p,kind='stable'),k)):
            binid[ids]=j
    out=[];total=0.
    for j in range(k):
        mask=binid==j
        if not mask.any():continue
        a=float(y[mask].mean());b=float(p[mask].mean());total+=float(mask.mean())*abs(a-b)
        out.append((j,int(mask.sum()),a,b))
    return float(total),out
rows=[];bins=[]
for split in ('validacion','prueba'):
 for model in (Y,M):
    a=D[(D.modelo==model)&(D.particion==split)]
    for group,threshold in EVALS:
        a0=a[a.score_raw>=threshold].copy()
        for state,field in [('Sin_calibrar','score_raw'),('Calibrado','score_calibrado')]:
            y=a0.correcta_IoU05.to_numpy();p=a0[field].to_numpy()
            for n in BINS:
                for typ in ('Igual_anchura','Igual_masa'):
                    value,details=ece(y,p,n,typ)
                    rows.append({'particion':split,'modelo':model,'poblacion':group,'umbral_RAW':threshold,
                                 'estado':state,'n_bins':n,'tipo_bins':typ,'n_detecciones':len(a0),
                                 'n_correctas':int(y.sum()),'ECE':value,
                                 'Brier':float(np.mean((p-y)**2)),
                                 'Log_loss':float(-np.mean(y*np.log(np.clip(p,1e-6,1))+(1-y)*np.log(np.clip(1-p,1e-6,1))))})
                    if split=='prueba' and n==10 and typ=='Igual_anchura' and group=='Operativa_raw>=0.25':
                        for bid,num,acc,conf in details:
                            bins.append({'modelo':model,'estado':state,'bin':bid,'n':num,
                                         'precision':acc,'confianza_media':conf})
S=pd.DataFrame(rows);B=pd.DataFrame(bins)
print('✅ Test ECE operativo (misma población para sin calibrar y calibrado):')
display(S[(S.particion=='prueba')&(S.poblacion=='Operativa_raw>=0.25')&
          (S.n_bins==10)&(S.tipo_bins=='Igual_anchura')][['modelo','estado','n_detecciones','ECE','Brier','Log_loss']])


## 4. Excel, figura en inglés, JSON y ZIP

In [ ]:
from matplotlib.patches import Rectangle
OUT=METRIC/'CERDOS_16_ECE_Sensibilidad_Operativo.xlsx'
BASE=FIG/'CERDOS_16_Confiabilidad_Operativa'
with pd.ExcelWriter(OUT,engine='openpyxl') as w:
    S.to_excel(w,sheet_name='Sensibilidad_ECE',index=False)
    B.to_excel(w,sheet_name='Bines_operativos',index=False)
    pd.DataFrame({'regla':['Filtro fijo sobre score_raw>=0.25, NO sobre score calibrado',
                            'ECE condicionado a detecciones retenidas; no incluye falsos negativos',
                            'Igual masa = grupos contiguos tras ordenar; empates de score se separan en caso necesario',
                            'Test ya inspeccionado previamente; NO equivale a validación externa',
                            'Selección de corte para visualización de sensibilidad, NO optimización del test']}).to_excel(w,sheet_name='Interpretacion',index=False)
plt.rcParams.update({'font.family':'DejaVu Serif','font.size':9,'pdf.fonttype':42})
fig,axes=plt.subplots(2,2,figsize=(11.4,8.8),sharex=True,sharey=True)
for i,model in enumerate((Y,M)):
 for j,state in enumerate(('Sin_calibrar','Calibrado')):
    ax=axes[i,j];b=B[(B.modelo==model)&(B.estado==state)].sort_values('confianza_media')
    ax.plot([0,1],[0,1],'--',color='#26282C',lw=1,label='Perfect calibration')
    ax.scatter(b.confianza_media,b.precision,s=150,marker='s',color='#668BAB',edgecolor='white',label='Observed precision',zorder=3)
    for _,row in b.iterrows():
        ax.vlines(float(row.confianza_media),float(row.confianza_media),float(row.precision),
                  color='#D47B76',lw=3,alpha=.8)
    val=float(S[(S.particion=='prueba')&(S.modelo==model)&(S.poblacion=='Operativa_raw>=0.25')&
                 (S.estado==state)&(S.n_bins==10)&(S.tipo_bins=='Igual_anchura')].ECE.iloc[0])
    ax.set(title=f'{model} — {"Before" if j==0 else "After"} calibration (ECE={val:.4f})',
           xlabel='Predicted confidence',ylabel='Observed precision')
    ax.set_xlim(0,1);ax.set_ylim(0,1);ax.grid(alpha=.15)
fig.suptitle('Reliability of retained pig detections (raw score >= 0.25)',fontsize=14)
fig.tight_layout(rect=(0,0,1,.97))
for ext in ('png','pdf','svg'):
    fig.savefig(str(BASE)+'.'+ext,dpi=450,bbox_inches='tight')
plt.show()
J=REPORT/'CERDOS_16_ECE_Sensibilidad.json'
J.write_text(json.dumps({'fase':'16','fecha_utc':datetime.now(timezone.utc).isoformat(),
      'comparacion_operativa':S[(S.particion=='prueba')&(S.n_bins==10)&
       (S.tipo_bins=='Igual_anchura')&(S.poblacion=='Operativa_raw>=0.25')].to_dict('records'),
      'limitaciones':['Población de detecciones es filtrada por RAW score >=.25 (la misma antes/después).',
       'ECE no representa falsos negativos ni incertidumbre píxel a píxel.','Resultados son post hoc sobre el test previamente inspeccionado.']},ensure_ascii=False,indent=2),encoding='utf8')
ZIP=EXPORT/'CERDOS_16_ECE_Sensibilidad_Resultados.zip'
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED) as z:
 for p in [OUT,J,*[Path(str(BASE)+'.'+e) for e in ('png','pdf','svg')]]:z.write(p,arcname=p.name)
assert zipfile.ZipFile(ZIP).testzip() is None
files.download(str(ZIP))
